<a href="https://colab.research.google.com/github/sdfbk/90-Days-Big-Data-Analysis-Challenge/blob/main/Partition_Bucketing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder\
        .appName("Partition and bucketing")\
        .getOrCreate()

In [3]:
df_netflix = spark.read.csv("/content/drive/MyDrive/netflix_titles.csv",header=True,inferSchema=True)
df_netflix.show()

+-------+-------+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+---------+--------------------+--------------------+
|show_id|   type|               title|            director|                cast|             country|        date_added|release_year|rating| duration|           listed_in|         description|
+-------+-------+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+---------+--------------------+--------------------+
|     s1|  Movie|Dick Johnson Is Dead|     Kirsten Johnson|                NULL|       United States|September 25, 2021|        2020| PG-13|   90 min|       Documentaries|As her father nea...|
|     s2|TV Show|       Blood & Water|                NULL|Ama Qamata, Khosi...|        South Africa|September 24, 2021|        2021| TV-MA|2 Seasons|International TV ...|After crossing pa...|
|     s3|TV Show|           Ganglan

In [6]:
df_netflix.count()

8809

In [7]:
df_netflix.printSchema()

root
 |-- show_id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- title: string (nullable = true)
 |-- director: string (nullable = true)
 |-- cast: string (nullable = true)
 |-- country: string (nullable = true)
 |-- date_added: string (nullable = true)
 |-- release_year: string (nullable = true)
 |-- rating: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- listed_in: string (nullable = true)
 |-- description: string (nullable = true)



In [9]:
df_netflix.write.mode("overwrite").partitionBy("country").csv("/content/drive/MyDrive/partition_by_country",header = True)

In [24]:
df_netflix.write.mode("overwrite").bucketBy(5,"show_id").saveAsTable("bucket_id")

In [28]:
# It shows Partition operation o/p
import os

# List contents of the directory
partitioned_path = "/content/drive/MyDrive/partition_by_country"

if os.path.exists(partitioned_path):
    print(os.listdir(partitioned_path))
else:
    print(f"Directory does not exist: {partitioned_path}")

['country=__HIVE_DEFAULT_PARTITION__', 'country= Ama K. Abebrese', 'country= Aziz Ansari', 'country= Chuck D.', 'country= Dominic Costa', 'country= Doug Plaut', 'country= Francesc Orella', 'country= Henri-Noël Tabary%22', 'country= James Toback%22', 'country= Justin %22%22Alyssa Edwards%22%22 Johnson', 'country= Lachion Buckingham', 'country= Leonardo Sbaraglia', 'country= Michael Cavalieri', 'country= Remilekun %22%22Reminisce%22%22 Safaru', 'country= Rob Morgan', 'country= Sophia Loren%22', 'country= Tantoo Cardinal', 'country= Theo Campbell%22', 'country= Tobechukwu %22%22iLLbliss%22%22 Ejiofor', 'country= plus Whitney Cummings gives suspect dating advice.%22', 'country=, France, Algeria', 'country=, South Korea', 'country=1944', 'country=Argentina', 'country=Argentina, Brazil, France, Poland, Germany, Denmark', 'country=Argentina, Chile', 'country=Argentina, Chile, Peru', 'country=Argentina, France', 'country=Argentina, France, United States, Germany, Qatar', 'country=Argentina, I

In [27]:
# It shows Partition operation o/p
import os

# Get the Spark warehouse directory
warehouse_dir = spark.conf.get("spark.sql.warehouse.dir")
print(f"Spark warehouse directory: {warehouse_dir}")

# Remove 'file:/' prefix if present for os module compatibility
local_warehouse_path = warehouse_dir.replace("file:", "")

# Construct the path to the bucketed table within the warehouse
table_path = os.path.join(local_warehouse_path, "bucket_id")

if os.path.exists(table_path):
    print(f"Contents of '{table_path}':")
    # List files directly, not subdirectories, to see the bucket files.
    # Often, bucketed files are directly under the table_path or a versioned subfolder.
    for root, dirs, files in os.walk(table_path):
        for name in files:
            print(os.path.join(root, name))
else:
    print(f"Spark table path does not exist: {table_path}")
    print("You might need to restart your Colab runtime if this is a new SparkSession and the warehouse directory was not initialized correctly.")

Spark warehouse directory: file:/content/spark-warehouse
Contents of '/content/spark-warehouse/bucket_id':
/content/spark-warehouse/bucket_id/part-00000-bb66a5d1-5465-49e0-85bf-c907ba42ee5d_00000.c000.snappy.parquet
/content/spark-warehouse/bucket_id/part-00000-bb66a5d1-5465-49e0-85bf-c907ba42ee5d_00001.c000.snappy.parquet
/content/spark-warehouse/bucket_id/_SUCCESS
/content/spark-warehouse/bucket_id/.part-00000-bb66a5d1-5465-49e0-85bf-c907ba42ee5d_00000.c000.snappy.parquet.crc
/content/spark-warehouse/bucket_id/.part-00000-bb66a5d1-5465-49e0-85bf-c907ba42ee5d_00003.c000.snappy.parquet.crc
/content/spark-warehouse/bucket_id/part-00000-bb66a5d1-5465-49e0-85bf-c907ba42ee5d_00002.c000.snappy.parquet
/content/spark-warehouse/bucket_id/._SUCCESS.crc
/content/spark-warehouse/bucket_id/.part-00000-bb66a5d1-5465-49e0-85bf-c907ba42ee5d_00002.c000.snappy.parquet.crc
/content/spark-warehouse/bucket_id/.part-00000-bb66a5d1-5465-49e0-85bf-c907ba42ee5d_00001.c000.snappy.parquet.crc
/content/spark-wa

This code first retrieves the Spark warehouse directory, then constructs the full path to your `bucket_id` table within that warehouse, and finally lists its contents. This will allow you to see the individual bucket files that Spark created.

In [ ]:
# dbutils.fs.ls("/content/drive/MyDrive/partition_by_country")  This is Databricks utility which is not available in colab